# Fine-tune the Hindi Piper voice on our own recordings

Runs on a free Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU).

Starts from Piper's male Hindi voice *rohan* (medium, 22,050 Hz) and fine-tunes it on the dataset made by
`bot/voice_training/prepare.py` (`<name>.zip` = `wavs/*.wav` + `metadata.csv`).

**Before you start:** upload `<name>.zip` to Google Drive in a folder `voice_training/`.
Everything (checkpoints, exported voice) is saved to that Drive folder, so a Colab disconnect loses nothing:
re-run the cells from the top and training resumes from the last checkpoint.

In [ ]:
# ---- settings -------------------------------------------------------------
NAME = "jasir"            # dataset zip is voice_training/<NAME>.zip on Drive; voice becomes hi_IN-<NAME>-medium
EXTRA_EPOCHS = 1000       # pilot (~8 min audio). For the full 30 min recording try 1500-2000.
BATCH_SIZE = 16           # T4 has 16 GB; lower to 8 if you get "CUDA out of memory"
SAVE_EVERY = 100          # epochs between checkpoints (~850 MB on Drive; only the newest is kept)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/voice_training'
RUN = f'{DRIVE}/{NAME}_run'
os.makedirs(RUN, exist_ok=True)
assert os.path.exists(f'{DRIVE}/{NAME}.zip'), f'upload {NAME}.zip to My Drive/voice_training/ first'

## 1. Install Piper training (about 3-5 min)

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
!pip install -q scikit-build cython
!test -d /content/piper1-gpl || git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!python3 setup.py build_ext --inplace
!pip install -e .[train]
!./build_monotonic_align.sh
!python3 -c "from piper import espeakbridge; print('espeakbridge successfully imported!')"

## 2. Dataset and the Hindi starting checkpoint

In [ ]:
!rm -rf /content/data && mkdir -p /content/data && unzip -q -o {DRIVE}/{NAME}.zip -d /content/data
!wc -l /content/data/metadata.csv && ls /content/data/wavs | wc -l && head -3 /content/data/metadata.csv

BASE = f'{DRIVE}/rohan_medium.ckpt'   # cached on Drive so it downloads only once (846 MB)
if not os.path.exists(BASE):
    !wget -q --show-progress -O "{BASE}" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/hi/hi_IN/rohan/medium/epoch%3D3190-step%3D309852.ckpt"
BASE_EPOCHS = 3190                     # the checkpoint's own epoch count; we train EXTRA_EPOCHS on top

## 3. Train

`--ckpt_path` resumes from a checkpoint (its epoch counter included), so `max_epochs` = 3190 + EXTRA_EPOCHS.
If a previous run of this cell saved checkpoints on Drive, it resumes from the newest one instead.

Watch `loss_gen_all` / `loss_disc_all` in the log; listen in step 4 while it trains (open a 2nd notebook tab) or after.

In [ ]:
import glob
ckpts = sorted(glob.glob(f'{RUN}/lightning_logs/version_*/checkpoints/*.ckpt'), key=os.path.getmtime)
START = ckpts[-1] if ckpts else BASE
print('starting from', START)

!PYTHONPATH=/content/piper1-gpl/src python3 -m piper.train fit \
  --data.voice_name "{NAME}" \
  --data.csv_path /content/data/metadata.csv \
  --data.audio_dir /content/data/wavs/ \
  --model.sample_rate 22050 \
  --data.espeak_voice "hi" \
  --data.cache_dir /content/cache/ \
  --data.config_path {RUN}/config.json \
  --data.batch_size {BATCH_SIZE} \
  --trainer.max_epochs {BASE_EPOCHS + EXTRA_EPOCHS} \
  --trainer.check_val_every_n_epoch {SAVE_EVERY} \
  --trainer.default_root_dir {RUN} \
  --trainer.callbacks+=lightning.pytorch.callbacks.ModelCheckpoint \
  --trainer.callbacks.every_n_epochs={SAVE_EVERY} \
  --trainer.callbacks.save_last=true \
  --ckpt_path "{START}"

## 4. Export the newest checkpoint and listen

In [ ]:
import glob, os, wave, sys
sys.path.append('/content/piper1-gpl/src')
from IPython.display import Audio, display
ckpt = sorted(glob.glob(f'{RUN}/lightning_logs/version_*/checkpoints/*.ckpt'), key=os.path.getmtime)[-1]
print('exporting', ckpt)
VOICE = f'hi_IN-{NAME}-medium'
!python3 -m piper.train.export_onnx --checkpoint "{ckpt}" --output-file /content/{VOICE}.onnx
!cp {RUN}/config.json /content/{VOICE}.onnx.json

from piper import PiperVoice
voice = PiperVoice.load(f'/content/{VOICE}.onnx', config_path=f'/content/{VOICE}.onnx.json')
TESTS = [
    "नमस्ते, क्या मेरी बात राहुल जी से हो रही है? मैं कृपाल, सोलरस्क्वेयर से बोल रहा हूँ।",
    "उस दिन हमारे कर्मचारी सुबह आठ से बारह बजे के बीच आएँगे। क्या यह समय आपके लिए ठीक रहेगा?",
    "और क्या वहाँ बिजली की सुविधा है?",
    "आपका समय शनिवार, अट्ठाईस नवंबर के लिए पक्का हो गया है। धन्यवाद, आपका दिन शुभ हो!",
    "मैं आपको कल शाम पाँच बजे दोबारा फ़ोन करूँगा।",   # not in the recording script: tests generalisation
]
for i, t in enumerate(TESTS):
    with wave.open(f'/content/test_{i}.wav', 'wb') as w:
        voice.synthesize_wav(t, w)
    print(t); display(Audio(f'/content/test_{i}.wav'))

## 5. Save the voice to Drive

In [ ]:
!cp /content/{VOICE}.onnx /content/{VOICE}.onnx.json {DRIVE}/
print(f'Download {VOICE}.onnx and {VOICE}.onnx.json from My Drive/voice_training/')